# DeepSeek-OCR MindSpore DEMO

基于 **MindSpore 2.7.0 + MindNLP 0.5.1** 的文本识别与结构化解析演示。

## 环境要求

| 组件 | 版本 |
|------|------|
| Python | 3.10 |
| MindSpore | 2.7.0 |
| MindNLP | 0.5.1 |
| transformers | 4.57.3 |
| Gradio | 6.1.0 |
| 硬件 | Ascend NPU 910B (65536MB HBM) |
| CANN | 8.2.RC2 |

In [ ]:
# Cell 1: 环境检查
import mindspore as ms
ms.set_context(device_target="Ascend", device_id=0)
print(f"MindSpore version: {ms.__version__}")

import mindnlp
print(f"MindNLP available")

import transformers
print(f"transformers version: {transformers.__version__}")

import gradio as gr
print(f"Gradio version: {gr.__version__}")

## 模型加载

使用 MindNLP 的 transformers 兼容接口加载 DeepSeek-OCR 模型。

**关键参数说明**：
- `_attn_implementation='eager'`: Ascend NPU 上兼容性最佳的注意力实现
- `trust_remote_code=True`: 加载模型自定义代码
- `use_safetensors=True`: 使用安全张量格式

In [ ]:
# Cell 2: 模型加载
import types
import mindnlp
import mindtorch
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

model_name = 'lvyufeng/DeepSeek-OCR'

print("加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("加载模型 (float32)...")
model = AutoModel.from_pretrained(
    model_name,
    _attn_implementation='eager',
    trust_remote_code=True,
    use_safetensors=True,
    device_map='auto'
)
model = model.eval()

print("合并 MoE 权重...")
model.combine_moe()

# NPU 不支持 scatter_add 用 one_hot 替代
def _patched_forward_for_moe(self, hidden_states):
    batch_size, sequence_length, hidden_dim = hidden_states.shape
    selected_experts, routing_weights = self.gate(hidden_states)
    n_experts = self.config.n_routed_experts
    routing_weights = routing_weights.to(hidden_states.dtype)
    one_hot = F.one_hot(selected_experts, n_experts).to(routing_weights.dtype)
    router_scores = (one_hot * routing_weights.unsqueeze(-1)).sum(dim=1)
    hidden_states = hidden_states.view(-1, hidden_dim)
    if self.config.n_shared_experts is not None:
        shared_expert_output = self.shared_experts(hidden_states)
    hidden_w1 = torch.matmul(hidden_states, self.w1)
    hidden_w3 = torch.matmul(hidden_states, self.w3)
    hidden_states = self.act(hidden_w1) * hidden_w3
    hidden_states = torch.bmm(hidden_states, self.w2) * torch.transpose(router_scores, 0, 1).unsqueeze(-1)
    final_hidden_states = hidden_states.sum(dim=0, dtype=hidden_states.dtype)
    if self.config.n_shared_experts is not None:
        hidden_states = final_hidden_states + shared_expert_output
    return hidden_states.view(batch_size, sequence_length, hidden_dim)

for layer in model.model.layers:
    if hasattr(layer.mlp, 'w1'):
        layer.mlp.forward = types.MethodType(_patched_forward_for_moe, layer.mlp)

print("模型加载完成!")

## 单张图片推理示例（非流式）

使用 `model.infer()` 方法进行标准推理，支持多种分辨率模式：

| 模式 | base_size | image_size | crop_mode | 适用场景 |
|------|-----------|------------|-----------|----------|
| Tiny | 512 | 512 | False | 快速预览 |
| Small | 640 | 640 | False | 一般文档 |
| Base | 1024 | 1024 | False | 高质量 |
| Large | 1280 | 1280 | False | 超高分辨率 |
| **Gundam** | **1024** | **640** | **True** | **推荐：精度速度最佳平衡** |

In [ ]:
# Cell 3: 单张图片推理（非流式）
import time
import os

# 准备测试图片
# 如果没有测试图片，可以从 HuggingFace 下载
image_file = 'image_ocr.jpg'
if not os.path.exists(image_file):
    import urllib.request
    url = 'https://hf-mirror.com/datasets/hf-internal-testing/fixtures_got_ocr/resolve/main/image_ocr.jpg'
    print(f"下载测试图片: {url}")
    urllib.request.urlretrieve(url, image_file)
    print("下载完成")

prompt = "<image>\nFree OCR. "
output_path = './output'
os.makedirs(output_path, exist_ok=True)

print("开始推理 (Gundam 模式)...")
t0 = time.time()

with mindtorch.no_grad():
    res = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        save_results=True,
        test_compress=True,
    )

elapsed = time.time() - t0
print(f"\n推理完成，总耗时: {elapsed:.2f}s")

# 显示结果
if os.path.exists(f'{output_path}/result.mmd'):
    with open(f'{output_path}/result.mmd', 'r') as f:
        print("\n识别结果:")
        print(f.read())

## 流式生成 + 时间统计

使用 `TextIteratorStreamer` 实现流式 token 输出，可以在生成过程中实时查看结果。

**核心思路**：
1. 从 `model.infer()` 中抽取图像预处理逻辑为独立函数
2. 用 `TextIteratorStreamer` 替换原始 `NoEOSTextStreamer`
3. 在独立线程中运行 `model.generate()`
4. 主线程通过 streamer 迭代获取 token 并统计时间

In [ ]:
# Cell 4: 流式生成 + 时间统计
import math
import importlib
from threading import Thread
from PIL import Image, ImageOps
from transformers import TextIteratorStreamer

# 导入模型辅助函数
_mod = importlib.import_module(type(model).__module__)
format_messages = _mod.format_messages
load_pil_images = _mod.load_pil_images
text_encode = _mod.text_encode
BasicImageTransform = _mod.BasicImageTransform
dynamic_preprocess = _mod.dynamic_preprocess

IMAGE_TOKEN = '<image>'
IMAGE_TOKEN_ID = 128815
PATCH_SIZE = 16
DOWNSAMPLE_RATIO = 4
BOS_ID = 0


def prepare_inputs(prompt_text, image_file, base_size, image_size, crop_mode):
    """从 model.infer() 中抽取的图像预处理逻辑。"""
    conversation = [
        {"role": "<|User|>", "content": prompt_text, "images": [image_file]},
        {"role": "<|Assistant|>", "content": ""},
    ]
    formatted_prompt = format_messages(conversations=conversation, sft_format='plain', system_prompt='')
    images = load_pil_images(conversation)

    image_transform = BasicImageTransform(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True)
    text_splits = formatted_prompt.split(IMAGE_TOKEN)

    images_list, images_crop_list, images_seq_mask = [], [], []
    tokenized_str = []
    images_spatial_crop = []

    for text_sep, image in zip(text_splits, images):
        tokenized_sep = text_encode(tokenizer, text_sep, bos=False, eos=False)
        tokenized_str += tokenized_sep
        images_seq_mask += [False] * len(tokenized_sep)

        if crop_mode:
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = [1, 1]
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(image)

            global_view = ImageOps.pad(image, (base_size, base_size),
                                       color=tuple(int(x * 255) for x in image_transform.mean))
            images_list.append(image_transform(global_view).to(model.dtype))
            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            if width_crop_num > 1 or height_crop_num > 1:
                for i in range(len(images_crop_raw)):
                    images_crop_list.append(image_transform(images_crop_raw[i]).to(model.dtype))

            num_queries = math.ceil((image_size // PATCH_SIZE) / DOWNSAMPLE_RATIO)
            num_queries_base = math.ceil((base_size // PATCH_SIZE) / DOWNSAMPLE_RATIO)

            tokenized_image = ([IMAGE_TOKEN_ID] * num_queries_base + [IMAGE_TOKEN_ID]) * num_queries_base
            tokenized_image += [IMAGE_TOKEN_ID]
            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([IMAGE_TOKEN_ID] * (num_queries * width_crop_num) + [IMAGE_TOKEN_ID]) * (
                    num_queries * height_crop_num)
            tokenized_str += tokenized_image
            images_seq_mask += [True] * len(tokenized_image)
        else:
            if image_size <= 640:
                image = image.resize((image_size, image_size))
            global_view = ImageOps.pad(image, (image_size, image_size),
                                       color=tuple(int(x * 255) for x in image_transform.mean))
            images_list.append(image_transform(global_view).to(model.dtype))
            images_spatial_crop.append([1, 1])

            num_queries = math.ceil((image_size // PATCH_SIZE) / DOWNSAMPLE_RATIO)
            tokenized_image = ([IMAGE_TOKEN_ID] * num_queries + [IMAGE_TOKEN_ID]) * num_queries
            tokenized_image += [IMAGE_TOKEN_ID]
            tokenized_str += tokenized_image
            images_seq_mask += [True] * len(tokenized_image)

    tokenized_sep = text_encode(tokenizer, text_splits[-1], bos=False, eos=False)
    tokenized_str += tokenized_sep
    images_seq_mask += [False] * len(tokenized_sep)
    tokenized_str = [BOS_ID] + tokenized_str
    images_seq_mask = [False] + images_seq_mask

    input_ids = torch.LongTensor(tokenized_str)
    images_seq_mask_t = torch.tensor(images_seq_mask, dtype=torch.bool)

    if len(images_list) == 0:
        images_ori = torch.zeros((1, 3, image_size, image_size))
        images_spatial_crop_t = torch.zeros((1, 2), dtype=torch.long)
        images_crop = torch.zeros((1, 3, base_size, base_size))
    else:
        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_t = torch.tensor(images_spatial_crop, dtype=torch.long)
        images_crop = torch.stack(images_crop_list, dim=0) if images_crop_list else torch.zeros((1, 3, base_size, base_size))

    return {
        'input_ids': input_ids.unsqueeze(0).cuda(),
        'images': [(images_crop.cuda(), images_ori.cuda())],
        'images_seq_mask': images_seq_mask_t.unsqueeze(0).cuda(),
        'images_spatial_crop': images_spatial_crop_t,
    }


# 流式推理
prompt_text = "<image>\nFree OCR. "

model.disable_torch_init()
inputs = prepare_inputs(prompt_text, image_file, base_size=1024, image_size=640, crop_mode=True)

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False)

generate_kwargs = dict(
    input_ids=inputs['input_ids'],
    images=inputs['images'],
    images_seq_mask=inputs['images_seq_mask'],
    images_spatial_crop=inputs['images_spatial_crop'],
    temperature=0.0,
    eos_token_id=tokenizer.eos_token_id,
    streamer=streamer,
    max_new_tokens=8192,
    no_repeat_ngram_size=20,
    use_cache=True,
)


def run_generate():
    with torch.no_grad():
        model.generate(**generate_kwargs)


thread = Thread(target=run_generate)
t_start = time.time()
thread.start()

first_token_time = None
token_count = 0
full_text = ""
STOP_STR = '<｜end▁of▁sentence｜>'

print("流式生成中...")
print("=" * 50)
for new_text in streamer:
    if first_token_time is None:
        first_token_time = time.time() - t_start
    token_count += 1
    full_text += new_text
    # 实时输出
    clean = new_text.replace(STOP_STR, '')
    if clean:
        print(clean, end='', flush=True)

thread.join()
total_time = time.time() - t_start

print("\n" + "=" * 50)
print(f"\n性能统计:")
print(f"  首 Token 延迟 (TTFT): {first_token_time:.3f}s")
print(f"  总 Token 数: {token_count}")
print(f"  总耗时: {total_time:.2f}s")
print(f"  生成速度: {token_count / total_time:.2f} tokens/s")
if token_count > 1 and first_token_time:
    decode_time = total_time - first_token_time
    print(f"  解码速度 (不含首 token): {(token_count - 1) / decode_time:.2f} tokens/s")

## 性能优化方案说明

### 已实施的优化

| # | 优化项 | 方法 | 效果 |
|---|--------|------|------|
| 1 | **MoE 权重合并** | `model.combine_moe()` | 将分散的专家权重合并为矩阵运算，减少内存访问次数，加速前向传播 |
| 2 | **scatter_add NPU 适配** | `F.one_hot` + 矩阵乘法 | 替换 NPU 不支持的 `scatter_add_ext` 算子，保证 MoE 合并后推理正确性 |
| 3 | **KV Cache** | `use_cache=True` | 缓存已计算的 Key/Value，避免自回归生成时重复计算所有位置的注意力 |
| 4 | **N-gram 去重** | `no_repeat_ngram_size=20` | 防止模型生成重复文本，提升有效 token 效率 |
| 5 | **Eager Attention** | `_attn_implementation='eager'` | 在 Ascend NPU 上比 Flash Attention 兼容性更好，避免算子不支持的问题 |

### 优化前后实测数据对比（Ascend 910B, Gundam 模式, 256 tokens）

| 配置 | TTFT | 生成速度 | 解码速度 | 加速比 |
|------|------|----------|----------|--------|
| **全部优化** (combine_moe + KV Cache) | 9.757s | **7.95 tok/s** | **11.34 tok/s** | 基线 |
| 关闭 MoE 合并 (无 combine_moe) | 10.805s | 1.68 tok/s | 2.29 tok/s | **4.95x 慢** |

> **结论**: `combine_moe()` 是最关键的优化项，使解码速度提升约 **5 倍**（2.29 → 11.34 tok/s）。

### 不同分辨率模式实测对比（256 tokens, 全部优化）

| 模式 | TTFT | 总耗时 | 生成速度 | 解码速度 | 适用场景 |
|------|------|--------|----------|----------|----------|
| Tiny (512) | **0.214s** | **23.36s** | **11.00 tok/s** | 11.06 tok/s | 快速预览、低分辨率 |
| Small (640) | 0.257s | 23.89s | 10.76 tok/s | 10.83 tok/s | 一般文档 |
| **Gundam (推荐)** | 9.757s | 32.33s | 7.95 tok/s | **11.34 tok/s** | 高精度 OCR |

> **说明**: Gundam 模式 TTFT 较高是因为 crop 模式需要处理多个图像切片（全局视图+局部切片），但解码速度与其他模式持平。Tiny 模式 TTFT 极低（0.2s），适合对延迟敏感的场景。

In [ ]:
# Cell 5: 不同分辨率模式对比（可选运行）
import time

results = {}
modes = {
    'Tiny':   {'base_size': 512,  'image_size': 512,  'crop_mode': False},
    'Small':  {'base_size': 640,  'image_size': 640,  'crop_mode': False},
    'Gundam': {'base_size': 1024, 'image_size': 640,  'crop_mode': True},
}

for mode_name, params in modes.items():
    print(f"\n{'='*50}")
    print(f"测试模式: {mode_name} (base={params['base_size']}, img={params['image_size']}, crop={params['crop_mode']})")
    print(f"{'='*50}")

    inputs = prepare_inputs(prompt_text, image_file, **params)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False)

    gen_kwargs = dict(
        input_ids=inputs['input_ids'],
        images=inputs['images'],
        images_seq_mask=inputs['images_seq_mask'],
        images_spatial_crop=inputs['images_spatial_crop'],
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        streamer=streamer,
        max_new_tokens=4096,
        no_repeat_ngram_size=20,
        use_cache=True,
    )

    def _gen():
        with torch.no_grad():
            model.generate(**gen_kwargs)

    thread = Thread(target=_gen)
    t0 = time.time()
    thread.start()

    ttft = None
    n_tokens = 0
    for text in streamer:
        if ttft is None:
            ttft = time.time() - t0
        n_tokens += 1
    thread.join()
    total = time.time() - t0

    results[mode_name] = {'ttft': ttft, 'tokens': n_tokens, 'total': total, 'tps': n_tokens / total}
    print(f"  TTFT: {ttft:.3f}s | Tokens: {n_tokens} | Total: {total:.2f}s | Speed: {n_tokens/total:.2f} tok/s")

print(f"\n{'='*60}")
print("对比汇总:")
print(f"{'模式':<10} {'TTFT':>8} {'Tokens':>8} {'总耗时':>8} {'速度':>12}")
print(f"{'-'*50}")
for name, r in results.items():
    print(f"{name:<10} {r['ttft']:>7.3f}s {r['tokens']:>8} {r['total']:>7.2f}s {r['tps']:>8.2f} tok/s")

## Gradio 交互 DEMO

启动完整的 Gradio Web 界面，支持：
- 图片上传
- 多种分辨率模式选择
- 多种任务类型（Free OCR / Markdown / 图表解析 / 文本定位）
- 流式文本输出
- 实时性能统计

魔乐社区链接：（待更新）


## 相关链接

- **魔乐社区 (ModelScope)**: [DeepSeek-OCR 模型页](https://modelers.cn/)
- **HuggingFace 模型**: [lvyufeng/DeepSeek-OCR](https://huggingface.co/lvyufeng/DeepSeek-OCR)
- **MindNLP 项目**: [GitHub - mindnlp](https://github.com/mindspore-lab/mindnlp)
- **MindSpore 官网**: [mindspore.cn](https://www.mindspore.cn/)